# AeroPulse Source Likelihood — 08: Gold Set

Build the human-review harness the plan calls the single most important
improvement, and state plainly what cannot be concluded until it is filled in.

## Objective

Improvement plan sections 27 and 28.

Section 28 calls a human-reviewed gold set "probably the single most
important improvement for source attribution", and section 27 explains
why: every metric in notebooks 05-07 measures agreement with labels the
same system generated. A PR-AUC of 0.9 there means the classifier
reproduces the heuristics from independent evidence. It says nothing about
whether the heuristics are right.

**No expert-reviewed source-attribution labels exist for this dataset, and
this notebook does not invent any.** Generating "gold" labels from the same
heuristics would produce a number that looks like validation and measures
nothing — the failure mode this notebook exists to prevent.

What it delivers instead:

1. a **stratified annotation sheet** over events, sampled so a reviewer sees
   the contested cases and not only the easy ones
2. the **scoring harness**, executed end to end against a synthetic
   stand-in so it is known to work the day real labels arrive
3. a measurement of **how much of the weak-label signal is heuristic
   tautology** — what the labels agree on when their own inputs are removed
4. an explicit **unmeasured** entry in the promotion gates, which is why
   nothing can be promoted yet


> **Scientific constraint, repeated in every notebook because it governs how
> the output may be used.** These are *likelihoods derived from heuristics*,
> not measured source attribution. No public ground-truth attribution
> catalogue exists for this dataset. A model that scores well here has
> reproduced its own labeling system — which is not the same as being right.

In [1]:
# ============================================================================
# SETUP — reusable logic lives in source_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import source_toolkit as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

cfg = st.load_config()
SOURCES = st.SOURCES
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood
sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
import joblib
events = pd.read_parquet(cfg.label_dir / "events.parquet")
cal_stage = cfg.stage_dir("calibration")
clf_stage = cfg.stage_dir("classifiers")
cal_report = json.loads((cal_stage / "calibration_report.json").read_text())
clf_report = json.loads((clf_stage / "classifier_report.json").read_text())
TRAINABLE = clf_report["trainable_sources"]
print("events:", f"{len(events):,}", "| sources:", TRAINABLE)

events: 44,563 | sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']


## 1. The annotation sheet

In [3]:
sheet = st.gold_set_sample(events, size=cfg.gold_set_size, random_state=cfg.random_state)
print(f"sampled {len(sheet):,} events for review")
print("\nstratification (weak-label dominant source x confidence tercile):")
print(pd.crosstab(sheet["dominant_source"], sheet["confidence_tercile"]).to_string())
print("\ncolumns a reviewer fills in:",
      [c for c in sheet.columns if c.startswith("expert") or c in ("reviewer", "review_notes")])
print()
print(sheet.head(5).to_string(index=False))

assert (sheet["expert_primary_source"] == "").all(), (
    "the sheet must ship unlabelled — prefilling it from the heuristics would "
    "make the gold set a copy of the system it audits"
)
print("\nPASS — every expert column is empty, as it must be.")

sampled 300 events for review

stratification (weak-label dominant source x confidence tercile):
confidence_tercile  low  medium  high
dominant_source                      
biomass_burning      20      20    20
dust                 20      20    20
industrial           20      20    20
regional_transport   20      20    20
traffic              20      20    20

columns a reviewer fills in: ['expert_primary_source', 'expert_secondary_sources', 'expert_confidence', 'reviewer', 'review_notes']

 event_id  location_id               event_start                 event_end  n_hours  peak_pm25  mean_pm25 dominant_source  dominant_prob confidence_tercile expert_primary_source expert_secondary_sources expert_confidence reviewer review_notes
    353.0      3409455 2026-03-24 15:30:00+00:00 2026-03-24 17:30:00+00:00        3  77.900002  72.366669 biomass_burning       0.924142                low                                                                                       
     12.0      34

### Review protocol

Written down because a gold set built inconsistently is worse than none.

- **Unit is the event**, not the hour (section 29). A reviewer judges a
  whole episode.
- **Multi-label**: record every source judged to be contributing, not one.
  The label schema is multi-label; the gold set must match it or it cannot
  score it.
- **"uncertain" is a valid answer** and must be recorded rather than
  forced, otherwise the gold set inherits the exact defect the old labeller
  had.
- Reviewers should see the **evidence** (fire maps, wind, co-pollutant
  traces) but **not** the model's prediction, or the labels become an
  agreement study.
- Target 100-500 events (section 28). Two reviewers on an overlapping
  subset gives an inter-rater agreement figure, without which a
  disagreement between model and gold cannot be attributed to either.

## 2. Exercise the scoring harness

In [4]:
# A synthetic stand-in for expert labels, used ONLY to prove the scoring
# path runs. It is deliberately noisy and deliberately not the weak label.
rng = np.random.default_rng(cfg.random_state)
mock = sheet.copy()
flip = rng.random(len(mock)) < 0.35
alternatives = np.array(TRAINABLE)
mock["expert_primary_source"] = np.where(
    flip, rng.choice(alternatives, size=len(mock)), mock["dominant_source"])
mock["expert_confidence"] = rng.choice(["high", "medium", "low"], size=len(mock))
print("MOCK labels only — 35% randomly reassigned. Any number below is a")
print("harness check, not a result.\n")

def score_gold(gold: pd.DataFrame) -> dict:
    scored = gold[gold["expert_primary_source"] != ""].copy()
    if scored.empty:
        return {"status": "no expert labels", "n": 0}
    agreement = float((scored["expert_primary_source"] == scored["dominant_source"]).mean())
    by_source = (
        scored.assign(correct=scored["expert_primary_source"] == scored["dominant_source"])
        .groupby("dominant_source", observed=True)["correct"]
        .agg(["size", "mean"]).rename(columns={"size": "n", "mean": "agreement"})
        .round(3)
    )
    return {"status": "scored", "n": int(len(scored)),
            "weak_label_agreement": round(agreement, 4),
            "by_source": by_source.to_dict(orient="index")}

result = score_gold(mock)
print(json.dumps(result, indent=2, default=str))
print("\nHarness runs end to end. Replace `mock` with the completed sheet.")

MOCK labels only — 35% randomly reassigned. Any number below is a
harness check, not a result.

{
  "status": "scored",
  "n": 300,
  "weak_label_agreement": 0.6733,
  "by_source": {
    "biomass_burning": {
      "n": 60,
      "agreement": 0.717
    },
    "dust": {
      "n": 60,
      "agreement": 0.7
    },
    "industrial": {
      "n": 60,
      "agreement": 0.667
    },
    "regional_transport": {
      "n": 60,
      "agreement": 0.633
    },
    "traffic": {
      "n": 60,
      "agreement": 0.65
    }
  }
}

Harness runs end to end. Replace `mock` with the completed sheet.


## 3. How much of the weak label is tautology?

A question that *can* be answered without expert labels. If the classifier
in notebook 05 reproduces the weak labels well, the interesting quantity is
how much of that comes from genuinely independent evidence rather than from
features that are proxies for the labeling-function inputs.

Compare the leak-safe model against one that is allowed to see the labeling
inputs. The gap is the part of the label that only its own inputs can
explain.

In [5]:
df = pd.read_parquet(cfg.feature_dir / "source_evidence_features.parquet")
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
labels = pd.read_parquet(cfg.label_dir / "weak_labels.parquet")
work = df.join(labels)

FEATURES = clf_report["features"]
leaky_extra = [c for c in st.LF_INPUT_COLUMNS
               if c in work.columns and pd.api.types.is_numeric_dtype(work[c])]
LEAKY = FEATURES + leaky_extra

train, valid, test = st.chrono_split(work, cfg.train_fraction, cfg.validation_fraction)
cap = cfg.max_train_rows or 300_000
train = train.iloc[(np.arange(min(cap, len(train))) * (len(train) / min(cap, len(train)))).astype(int)]

rows = []
for source in TRAINABLE:
    safe = st.SourceClassifier.fit(train, FEATURES, source, kind=clf_report["winner_family"])
    leak = st.SourceClassifier.fit(train, LEAKY, source, kind=clf_report["winner_family"])
    m_safe = st.calibration_metrics(test[f"y_{source}"].to_numpy(), safe.raw_proba(test))
    m_leak = st.calibration_metrics(test[f"y_{source}"].to_numpy(), leak.raw_proba(test))
    rows.append({"source": source,
                 "leak_safe_pr_auc": round(m_safe["pr_auc"], 4),
                 "label_inputs_visible_pr_auc": round(m_leak["pr_auc"], 4),
                 "gap": round(m_leak["pr_auc"] - m_safe["pr_auc"], 4),
                 "pct_of_label_only_its_inputs_explain":
                     round(100 * (m_leak["pr_auc"] - m_safe["pr_auc"]) /
                           max(m_leak["pr_auc"], 1e-6), 1)})
    print(f"  {source:<20} leak-safe {m_safe['pr_auc']:.4f}   "
          f"with label inputs {m_leak['pr_auc']:.4f}   gap {m_leak['pr_auc'] - m_safe['pr_auc']:+.4f}")

tautology = pd.DataFrame(rows)
print()
print(tautology.to_string(index=False))

  biomass_burning      leak-safe 0.0106   with label inputs 0.9994   gap +0.9888


  traffic              leak-safe 0.6210   with label inputs 1.0000   gap +0.3790


  industrial           leak-safe 0.1448   with label inputs 1.0000   gap +0.8551


  dust                 leak-safe 0.7195   with label inputs 1.0000   gap +0.2805


  regional_transport   leak-safe 0.0144   with label inputs 0.9998   gap +0.9854

            source  leak_safe_pr_auc  label_inputs_visible_pr_auc    gap  pct_of_label_only_its_inputs_explain
   biomass_burning            0.0106                       0.9994 0.9888                                  98.9
           traffic            0.6210                       1.0000 0.3790                                  37.9
        industrial            0.1448                       1.0000 0.8551                                  85.5
              dust            0.7195                       1.0000 0.2805                                  28.0
regional_transport            0.0144                       0.9998 0.9854                                  98.6


The gap is an upper bound on how much the weak label is a restatement of
its own inputs. A **small** gap means independent evidence explains the
label nearly as well as the rule that made it — the source signal is real
in the data. A **large** gap means the label is mostly its own definition,
and the leak-safe classifier is right to score lower.

## 4. Save

In [6]:
stage = cfg.stage_dir("gold_set")
sheet.to_csv(stage / "gold_set_annotation_sheet.csv", index=False)
tautology.to_csv(stage / "tautology_analysis.csv", index=False)

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "gold_set_status": "UNLABELLED — annotation sheet generated, no expert review yet",
    "gold_set_labelled": False,
    "n_events_sampled": int(len(sheet)),
    "stratification": "weak-label dominant source x confidence tercile",
    "harness_check": result,
    "tautology_analysis": tautology.to_dict(orient="records"),
    "protocol": [
        "unit is the event, not the hour",
        "multi-label: record every contributing source",
        "'uncertain' is a valid and required option",
        "reviewers see evidence, never the model prediction",
        "two reviewers on an overlapping subset for inter-rater agreement",
        "target 100-500 events",
    ],
    "caveat": (
        "Until this sheet is reviewed, no metric in this pipeline measures "
        "source-attribution accuracy. Everything measures agreement with the "
        "weak-labeling system."
    ),
}
(stage / "gold_set_report.json").write_text(json.dumps(report, indent=2, default=str))
print("08 COMPLETE ->", stage)
print("\nGOLD SET STATUS: unlabelled. The promotion gate for it fails, and it")
print("should — the model has not been measured against anything but itself.")
print("Next: 09_source_model_packaging.ipynb")

08 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/artifacts/source/gold_set

GOLD SET STATUS: unlabelled. The promotion gate for it fails, and it
should — the model has not been measured against anything but itself.
Next: 09_source_model_packaging.ipynb


## Findings

**No gold set exists, so nothing in this pipeline has measured source
attribution accuracy.** That sentence is the finding. The annotation sheet
(stratified over dominant source x confidence tercile), the review protocol
and the scoring harness are all delivered and the harness is exercised end
to end against deliberately noisy mock labels — but every expert column
ships empty, asserted in code, because filling it from the heuristics would
turn the audit into a copy of the thing being audited.

**The tautology analysis is the substantive result, and it is the most
important table in this pipeline.** Comparing a leak-safe classifier against
one allowed to see the labeling-function inputs measures how much of each
weak label is a restatement of its own definition:

| source | leak-safe PR-AUC | with label inputs | % of the label only its own inputs explain |
|---|---|---|---|
| dust | 0.720 | 1.000 | **28.0%** |
| traffic | 0.621 | 1.000 | **37.9%** |
| industrial | 0.145 | 1.000 | 85.5% |
| regional_transport | 0.014 | 1.000 | **98.6%** |
| biomass_burning | 0.011 | 0.999 | **98.9%** |

Read the right-hand column as a ranking of how real each hypothesis is in
this dataset.

**Dust and traffic are largely learnable from independent evidence.** Around
two thirds of what defines those labels is recoverable from weather,
calendar, terrain and CAMS CO/O3 — evidence the labeling functions never
touched. Those two hypotheses have some claim to be measuring something
about the atmosphere.

**Biomass burning and regional transport are, on this evidence, almost
entirely definitional.** 98.9% and 98.6% of what makes those labels
positive is explained only by the columns that define them and by nothing
else in the dataset. Strip out `upwind_fire_frp_50km` and biomass burning
becomes very nearly unpredictable (PR-AUC 0.011).

That is not a modelling failure — it is close to a tautology by
construction. "Biomass burning" here *means* "there is upwind fire radiative
power and PM2.5 is high". A model denied both terms has nothing left, and a
model given them scores 0.999 while learning nothing at all. Notebook 06's
apparently impressive 10.8x biomass lift has to be read against this.

The practical consequence: for biomass burning and regional transport, the
weak label and the evidence are the same object, so **the classifier adds
essentially nothing over the labeling function** and AeroPulse would be
better served by serving the labeling function directly, with its
reliability weight, than by a model that pretends to have learned it.

That conclusion could not have been reached without the gold set being
absent and honestly marked as absent — which is the argument for building
one, exactly as plan section 28 says.